# ddldelta: generating Jinja-templated schemachange scaffolding

The previous notebooks diffed DDL directories into migrations. This one is
about a different, related need: you already maintain a set of hand-written
schemachange `A__`/`R__` scripts that call a shared Jinja macro to
parametrize a family of COPY INTO workloads, and you would rather
*generate* the per-table scripts from the current schema than hand-copy one
per table.

There are three separate Jinja questions in play here, all covered in the
README's ["Using ddldelta alongside schemachange's Jinja
features"](../README.md#using-ddldelta-alongside-schemachanges-jinja-features)
section:

1. **Output coexistence** — ddldelta's own generated `V__` migrations are
   Jinja-*inert* plain SQL; hand-written, Jinja-heavy `A__`/`R__` scripts and
   macro modules live untouched in the same `root-folder`.
2. **Templated input DDL is unsupported** — a vendor DDL file containing
   `{{ }}` fails parsing loudly rather than being silently mis-parsed.
3. **Templated output** is what this notebook demonstrates: the optional
   `JinjaRenderer` (the `jinja` extra) turns a schema into a set of files a
   *user-supplied* template controls.

In [1]:
import tempfile
from pathlib import Path

work = Path(tempfile.mkdtemp(prefix="ddldelta_jinja_"))
print("workspace:", work.name)

workspace: ddldelta_jinja_7gr_vyme


## The current schema

Two tables, as plain MySQL `CREATE TABLE` DDL — the same kind of vendor
delivery the earlier notebooks diffed. This time there is nothing to diff:
the goal is scaffolding *from* the current shape, not a migration *between*
two shapes.

In [2]:
ddl_dir = work / "schema"
ddl_dir.mkdir()

(ddl_dir / "customer.sql").write_text("""\
CREATE TABLE `customer` (
  `id` int(10) unsigned NOT NULL,
  `name` varchar(200) NOT NULL,
  `email` varchar(200) NOT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")

(ddl_dir / "order_line").with_suffix(".sql").write_text("""\
CREATE TABLE `order_line` (
  `id` int(10) unsigned NOT NULL,
  `customer_id` int(10) unsigned NOT NULL,
  `sku` varchar(50) NOT NULL,
  `qty` int(10) unsigned NOT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")

print(sorted(p.name for p in ddl_dir.glob("*.sql")))

['customer.sql', 'order_line.sql']


## The templates

`macros.sql.j2` defines one macro, `copy_into`, that renders a parametrized
COPY INTO statement — the kind of thing that today probably exists once by
hand and gets called from several `A__`/`R__` scripts. `copy_script.sql.j2`
is the per-table script shape: it imports the macro module and calls it
with that table's columns and stage path.

This is exactly the macro-module pattern from the README's coexistence
section — `JinjaRenderer` only generates the calling scripts, the macro
module itself is a plain file in the same `searchpath` a human could edit
by hand just as easily.

In [3]:
templates_dir = work / "templates"
templates_dir.mkdir()

(templates_dir / "macros.sql.j2").write_text("""\
{% macro copy_into(table, columns, stage) -%}
COPY INTO "{{ table }}" ({{ columns | join(', ') }})
  FROM @{{ stage }}
  FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY = '"')
  ON_ERROR = 'ABORT_STATEMENT';
{%- endmacro %}
""", encoding="utf-8")

(templates_dir / "copy_script.sql.j2").write_text("""\
{% import "macros.sql.j2" as macros %}
-- generated scaffolding for {{ table.name }} ({{ label }})
{{ macros.copy_into(table.name, columns, "acme_stage/" ~ table.name) }}
""", encoding="utf-8")

print(sorted(p.name for p in templates_dir.glob("*.j2")))

['copy_script.sql.j2', 'macros.sql.j2']


## Parsing the schema

`parse_path` is the same building block the diff-based workflow uses — a
`Schema` here is just the destination for `render_schema`, not one side of
a diff.

In [4]:
from ddldelta.ddl_parser import parser_for
from ddldelta.sources import parse_path

parse_ddl = parser_for("mysql")
schema = parse_path(parse_ddl, ddl_dir)
print(sorted(schema))

['customer', 'order_line']


## `render_schema`: one COPY INTO script per table

`JinjaRenderer.render_schema` walks every table in the schema (sorted by
name) and renders `template` once per table, with a context built from that
table — no diff, no plan, just the current shape. `filename_template` is
itself a small Jinja template, rendered with the same context, so the
output filename can depend on the table name.

In [5]:
from ddldelta.render.jinja import JinjaRenderer

copy_dir = work / "schemachange" / "copy"
renderer = JinjaRenderer(
    target_dir=copy_dir,
    searchpath=templates_dir,
    template="copy_script.sql.j2",
    filename_template="A__copy_{{ table.name }}.sql",
)
written = renderer.render_schema(schema, label="2026Q3")
print(sorted(str(p.relative_to(work)) for p in written))

['schemachange\\copy\\A__copy_customer.sql', 'schemachange\\copy\\A__copy_order_line.sql']


In [6]:
print((copy_dir / "A__copy_order_line.sql").read_text(encoding="utf-8"))


-- generated scaffolding for order_line (2026Q3)
COPY INTO "order_line" (id, customer_id, sku, qty)
  FROM @acme_stage/order_line
  FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY = '"')
  ON_ERROR = 'ABORT_STATEMENT';



## A quick look at the other mode: `render(plan)`

`JinjaRenderer` also satisfies the `Renderer` protocol the migration
renderers implement (`render(plan) -> tuple[Path, ...]`), for a template
that wants diff/fuse information — `critical`, the raw `statements`, the
`comparison` string — rather than a bare schema. Presentation of a critical
table (comment it out? highlight it? both?) becomes the template's call
instead of being fixed by the renderer, unlike `SchemachangeRenderer` /
`FlywayRenderer`.

In [7]:
from ddldelta.plan import baseline_plan

(templates_dir / "migration.sql.j2").write_text("""\
-- {{ comparison }} (critical={{ critical }})
{% for s in statements %}{{ s }}
{% endfor %}""", encoding="utf-8")

plan = baseline_plan("1", schema)
plan_renderer = JinjaRenderer(
    target_dir=work / "migrations",
    searchpath=templates_dir,
    template="migration.sql.j2",
    filename_template="V{{ label }}__{{ table.table }}.sql",
)
plan_written = plan_renderer.render(plan)
print(sorted(str(p.relative_to(work)) for p in plan_written))
print((work / "migrations" / "V1__customer.sql").read_text(encoding="utf-8"))

['migrations\\V1__customer.sql', 'migrations\\V1__order_line.sql']
-- (baseline) -> 1 (critical=False)
CREATE TABLE "customer" (
    "id" INT NOT NULL,
    "name" VARCHAR(200) NOT NULL,
    "email" VARCHAR(200) NOT NULL
);



## Overwrite semantics and the checksum caveat

Unlike `SchemachangeRenderer`/`FlywayRenderer`, `JinjaRenderer` has no
exists-guard: every call overwrites its target files. That is the right
default for regenerable scaffolding — rerun `render_schema` after adding a
column and the COPY INTO list picks it up automatically, no
`GenerationError` to work around. It also means the responsibility flips:
if `target_dir` is a folder schemachange also checksums (an `R__` folder,
say), regenerating after that script has been deployed changes its
rendered bytes and re-triggers it on the next deploy — a consequence to
design around deliberately, not a bug in the renderer. See decision D12 in
`docs/decisions.md` for the full reasoning behind treating templated output
this way.